In [1]:
import sys
import pandas as pd
import numpy as np
import torch
import optuna
from datetime import datetime

from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments    
)

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt

sys.path.append('../src')

from dataset import RepositorioDados
from models.har import HarModel
from models.transformer import compute_metrics, evaluate_and_visualize

# Detecta o dispositivo e a precisão usada nas operações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device} | Precision: {dtype}")

# Seed para resultados reproduzíveis
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

c:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu | Precision: torch.float32


In [25]:
# ==================== CONFIGURAÇÕES PRÉ-TREINAMENTO ====================
TIMESTAMP_COLUMN = 'timestamp'  # Coluna com timestamps em ms

# Tamanhos das janelas: histórico 7 dias, previsão 1 dia (dados diários)
CONTEXT_LENGTH = 512         # Janela histórica: x dias passados
FORECAST_HORIZON = 1        # Prever: x dias à frente

TRAIN_FRAC, VALID_FRAC = 0.7, 0.1  # Frações treino/validação/teste

# Hyperparâmetros do modelo
PATCH_LENGTH = 1            # Tamanho do patch (1=sem patchificação, mantém cada dia)
BATCH_SIZE = 32             # Samples por batch (reduzir se GPU memory limitada)
NUM_WORKERS = 0             # Workers para data loading (0 em Windows)
EPOCHS = 50                 # Reduzido: 50→30 (volatilidade tem ciclos curtos)
LEARNING_RATE = 1e-4        # Taxa de aprendizado

In [26]:
repo = RepositorioDados()

In [27]:
FEATURES = ["Vol_lag_1", "Vol_lag_2", "Vol_lag_3"]
TARGET_COLUMN = ['Vol']
ID_COLUMNS = []

In [28]:
tsp, train_ds, valid_ds, test_ds = repo.executar(
    timestamp_col=TIMESTAMP_COLUMN,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    context_length=CONTEXT_LENGTH,
    features=FEATURES,
    target=TARGET_COLUMN,
    id_cols=ID_COLUMNS,
    forecast_horizon=FORECAST_HORIZON,
    use_mean_features=False,
    lags=3
)

Carregando dados de C:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1279 amostras | Val: 256 | Teste: 512


In [29]:
train_ds.datasets[0].data_df

,timestamp,Vol,group
0,2017-11-30,2.618216,0
1,2017-12-01,0.700721,0
2,2017-12-02,0.182059,0
3,2017-12-03,1.475025,0
4,2017-12-04,0.422412,0
...,...,...,...
1786,2022-10-21,-0.345558,0
1787,2022-10-22,-0.414484,0
1788,2022-10-23,-0.384091,0
1789,2022-10-24,-0.372276,0


In [30]:
# 1. Features
FEATURE_COMBINATIONS = {
    1: ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"],
    3: ["Vol_lag_1", "Vol_lag_2", "Vol_lag_3"],
}

# 2. Janelas temporais
CONTEXT_LENGTHS = [256, 512]
FORECAST_HORIZONS = [1]  # Manter fixo por enquanto

# 3. Hiperparâmetros do modelo
MODEL_PARAMS = {
    'd_model': [64, 128],
    'num_attention_heads': [8, 16],
    'num_hidden_layers': [2, 3],
    'ffn_dim': [256, 512],
    'dropout': [0.1, 0.2],
    'patch_length': [1, 16],
}

# 4. Hiperparâmetros de treinamento
TRAINING_PARAMS = {
    'learning_rate': [1e-4, 5e-4],
    'batch_size': [32, 64],
}

# Configurações fixas
FIXED_PARAMS = {
    'epochs': 10,  # Reduzido para grid search
    'early_stopping_patience': 3,
    'num_workers': 0,
    'train_frac': 0.7,
    'valid_frac': 0.1,
}

print("Espaços de busca definidos")
print(f"Total de combinações de features: {len(FEATURE_COMBINATIONS)}")
print(f"Total de combinações de context_length: {len(CONTEXT_LENGTHS)}")
print(f"Total de combinações de modelo: {np.prod([len(v) for v in MODEL_PARAMS.values()])}")
print(f"Total de combinações de treinamento: {np.prod([len(v) for v in TRAINING_PARAMS.values()])}")
total = len(FEATURE_COMBINATIONS) * len(CONTEXT_LENGTHS) * np.prod([len(v) for v in MODEL_PARAMS.values()]) * np.prod([len(v) for v in TRAINING_PARAMS.values()])
print(f"\n🔍 Total de experimentos: {int(total)}")

Espaços de busca definidos
Total de combinações de features: 2
Total de combinações de context_length: 2
Total de combinações de modelo: 64
Total de combinações de treinamento: 4

🔍 Total de experimentos: 1024


In [ ]:
def run_experiment(
    features,
    context_length,
    forecast_horizon,
    d_model,
    num_attention_heads,
    num_hidden_layers,
    ffn_dim,
    dropout,
    patch_length,
    learning_rate,
    batch_size,
    experiment_id,
    use_mean_features,
    lags
):
    """
    Executa um experimento completo com os parâmetros fornecidos.
    Retorna um dicionário com os resultados.
    
    trial: Optuna trial para pruning (opcional)
    """
    print(f"\n{'='*80}")
    print(f"🧪 EXPERIMENTO {experiment_id}")
    print(f"{'='*80}")
    print(f"Features: {features}")
    print(f"Context Length: {context_length}")
    print(f"d_model: {d_model}, heads: {num_attention_heads}, layers: {num_hidden_layers}")
    print(f"LR: {learning_rate}, Batch: {batch_size}, Patch: {patch_length}")
    print(f"{'='*80}\n")
    
    try:
        # Escolher configuração (rápida ou completa)
        config_params = FIXED_PARAMS
        
        # 1. Preparar dados
        tsp, train_ds, valid_ds, test_ds = repo.executar(
            timestamp_col=TIMESTAMP_COLUMN,
            train_frac=config_params['train_frac'],
            valid_frac=config_params['valid_frac'],
            context_length=context_length,
            features=features,
            target=TARGET_COLUMN,
            id_cols=ID_COLUMNS,
            forecast_horizon=forecast_horizon,
            use_mean_features=use_mean_features,
            lags=lags
        )
        
        # 2. Configurar modelo
        config = PatchTSTConfig(
            do_mask_input=False,
            context_length=context_length,
            patch_length=patch_length,
            num_input_channels=len(TARGET_COLUMN),
            patch_stride=patch_length,
            prediction_length=forecast_horizon,
            d_model=d_model,
            num_attention_heads=num_attention_heads,
            num_hidden_layers=num_hidden_layers,
            ffn_dim=ffn_dim,
            dropout=dropout,
            head_dropout=dropout,
            pooling_type=None,
            channel_attention=True,
            scaling='std',
            loss='mse',
            pre_norm=True,
            norm_type='batchnorm',
        )
        
        model = PatchTSTForPrediction(config=config).to(device).to(dtype)
        
        # 3. Configurar treinamento
        train_args = TrainingArguments(
            output_dir="./grid_search_temp",
            overwrite_output_dir=True,
            learning_rate=learning_rate,
            num_train_epochs=config_params['epochs'],
            do_eval=True,
            eval_strategy="epoch",
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            dataloader_num_workers=config_params['num_workers'],
            save_strategy="no",
            logging_strategy="epoch",
            logging_dir=None,
            load_best_model_at_end=False,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            label_names=["future_values"],
            report_to="none",
            fp16=(dtype == torch.float16),
            bf16=(dtype == torch.bfloat16),
        )
        
        early_stopping = EarlyStoppingCallback(
            early_stopping_patience=config_params['early_stopping_patience'],
            early_stopping_threshold=0.001
        )
        
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=valid_ds,
            compute_metrics=compute_metrics,
            callbacks=[early_stopping]
        )
        
        # 4. Treinar
        train_result = trainer.train()
        
        # 5. Avaliar no conjunto de validação
        eval_result = trainer.evaluate()
        
        # 6. Coletar métricas
        result = {
            'experiment_id': experiment_id,
            'features': str(features),
            'num_features': len(features),
            'context_length': context_length,
            'forecast_horizon': forecast_horizon,
            'd_model': d_model,
            'num_attention_heads': num_attention_heads,
            'num_hidden_layers': num_hidden_layers,
            'ffn_dim': ffn_dim,
            'dropout': dropout,
            'patch_length': patch_length,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'train_loss': train_result.training_loss,
            'eval_loss': eval_result['eval_loss'],
            'eval_MSE': eval_result.get('eval_MSE', None),
            'eval_MAE': eval_result.get('eval_MAE', None),
            'eval_RMSE': eval_result.get('eval_RMSE', None),
            'eval_MAPE': eval_result.get('eval_MAPE', None),
            'epochs_trained': train_result.global_step // len(train_ds) * batch_size,
            'status': 'success'
        }
        
        print(f"✅ Experimento {experiment_id} concluído!")
        print(f"   Val Loss: {eval_result['eval_loss']:.6f} | RMSE: {result['eval_RMSE']:.6f}")
        
        # Limpar memória
        del model, trainer, train_ds, valid_ds, test_ds, tsp
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return result
        
    except Exception as e:
        print(f"❌ Erro no experimento {experiment_id}: {str(e)}")
        return {
            'experiment_id': experiment_id,
            'features': str(features),
            'context_length': context_length,
            'd_model': d_model,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'status': 'failed',
            'error': str(e)
        }

print("✓ Função run_experiment definida (com modo rápido)")

✓ Função run_experiment definida (com modo rápido)


In [32]:
def optuna_objective(trial):
    """
    Função objetivo para Optuna.
    Retorna a métrica a ser MINIMIZADA (eval_loss ou RMSE).
    """
    # Sugere hiperparâmetros a partir dos espaços definidos na célula 6
    features_idx = trial.suggest_categorical('features_idx', list(FEATURE_COMBINATIONS.keys()))
    features = FEATURE_COMBINATIONS[features_idx]

    context_length = trial.suggest_categorical('context_length', CONTEXT_LENGTHS)
    forecast_horizon = trial.suggest_categorical('forecast_horizon', FORECAST_HORIZONS)

    d_model = trial.suggest_categorical('d_model', MODEL_PARAMS['d_model'])
    num_attention_heads = trial.suggest_categorical('num_attention_heads', MODEL_PARAMS['num_attention_heads'])
    num_hidden_layers = trial.suggest_categorical('num_hidden_layers', MODEL_PARAMS['num_hidden_layers'])
    ffn_dim = trial.suggest_categorical('ffn_dim', MODEL_PARAMS['ffn_dim'])
    dropout = trial.suggest_categorical('dropout', MODEL_PARAMS['dropout'])
    patch_length = trial.suggest_categorical('patch_length', MODEL_PARAMS['patch_length'])

    learning_rate = trial.suggest_categorical('learning_rate', TRAINING_PARAMS['learning_rate'])
    batch_size = trial.suggest_categorical('batch_size', TRAINING_PARAMS['batch_size'])

    # Executar experimento
    result = run_experiment(
        features=features,
        context_length=context_length,
        forecast_horizon=forecast_horizon,
        d_model=d_model,
        num_attention_heads=num_attention_heads,
        num_hidden_layers=num_hidden_layers,
        ffn_dim=ffn_dim,
        dropout=dropout,
        patch_length=patch_length,
        learning_rate=learning_rate,
        batch_size=batch_size,
        experiment_id=trial.number,
        use_mean_features=True if features_idx == 1 else False,
        lags=features_idx
    )

    # Salvar resultado completo
    trial.set_user_attr('full_result', result)

    # Se falhou, retornar valor alto
    if result['status'] == 'failed':
        return float('inf')

    # Retornar métrica a minimizar
    return result['eval_RMSE']  # ou 'eval_loss'

In [33]:
study = optuna.create_study(
    direction='minimize',  # Minimizar RMSE
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name='patchtst_optimization'
)

[I 2026-02-10 00:31:26,221] A new study created in memory with name: patchtst_optimization


In [34]:
start_time = datetime.now()

# Executar otimização
study.optimize(
    optuna_objective,
    n_trials=50,  # ⚡ REDUZIDO: 200 → 50
    show_progress_bar=True,
    callbacks=[
        lambda study, trial: study.trials_dataframe().to_csv(
            'optuna_results_partial.csv', index=False
        ) if trial.number % 5 == 0 else None
    ],
    gc_after_trial=True  # ⚡ NOVO: Libera memória após cada trial
)

end_time = datetime.now()
duration = end_time - start_time

print(f"\n{'='*80}")
print(f"✅ Busca Optuna concluída!")
print(f"⏱️ Tempo total: {duration}")
print(f"🏆 Melhor RMSE: {study.best_value:.6f}")
print(f"{'='*80}\n")

# Extrair resultados
results = [trial.user_attrs.get('full_result') for trial in study.trials 
            if 'full_result' in trial.user_attrs]

# Salvar resultados
df_results = pd.DataFrame(results)
df_results.to_csv('optuna_results.csv', index=False)

# Mostrar melhores parâmetros
print("🏆 MELHORES HIPERPARÂMETROS (Optuna):")
print("="*80)
for key, value in study.best_params.items():
    print(f"{key:25s}: {value}")
print("="*80)

# Visualização Optuna
try:
    from optuna.visualization import plot_optimization_history, plot_param_importances
    
    # Histórico de otimização
    fig1 = plot_optimization_history(study)
    fig1.write_image('optuna_history.png')
    fig1.show()
    
    # Importância dos parâmetros
    fig2 = plot_param_importances(study)
    fig2.write_image('optuna_importance.png')
    fig2.show()
    
    print("📊 Gráficos salvos: optuna_history.png, optuna_importance.png")
except Exception as e:
    print(f"⚠️ Não foi possível gerar visualizações Optuna: {e}")
    print("   Instale: pip install plotly kaleido")

  0%|          | 0/50 [00:00<?, ?it/s]


🧪 EXPERIMENTO 0
Features: ['Vol_lag_1', 'Vol_lag_2', 'Vol_lag_3']
Context Length: 256
d_model: 64, heads: 16, layers: 3
LR: 0.0005, Batch: 32, Patch: 16

Carregando dados de C:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1535 amostras | Val: 256 | Teste: 512


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Mse,Mae,Rmse,Mape
1,0.608200,0.029116,0.029116,0.091899,0.170635,131.918526
2,0.581500,0.025920,0.025920,0.077936,0.160998,123.395967
3,0.583300,0.028090,0.028090,0.076312,0.167601,131.401694
4,0.567000,0.027376,0.027376,0.077225,0.165456,94.406712
5,0.560800,0.026907,0.026907,0.077735,0.164034,99.264359


Best trial: 0. Best value: 0.164034:   2%|▏         | 1/50 [00:19<15:45, 19.29s/it]

...

Epoch,Training Loss,Validation Loss,Mse,Mae,Rmse,Mape
1,0.664800,0.030430,0.030430,0.087701,0.174443,54.827821
2,0.592700,0.035617,0.035617,0.093829,0.188724,83.946347
3,0.594500,0.029458,0.029458,0.088745,0.171634,69.282556
4,0.578100,0.027126,0.027126,0.084897,0.164701,59.537351
5,0.554600,0.028443,0.028443,0.078826,0.168652,55.578113
6,0.536600,0.028252,0.028252,0.084633,0.168085,61.092782
7,0.525700,0.027034,0.027034,0.075976,0.164420,56.471157


Best trial: 48. Best value: 0.156096:  98%|█████████▊| 49/50 [2:01:47<01:58, 118.13s/it]

✅ Experimento 49 concluído!
   Val Loss: 0.027034 | RMSE: 0.164420
[I 2026-02-10 02:33:14,118] Trial 49 finished with value: 0.164419721939663 and parameters: {'features_idx': 1, 'context_length': 256, 'forecast_horizon': 1, 'd_model': 128, 'num_attention_heads': 8, 'num_hidden_layers': 2, 'ffn_dim': 256, 'dropout': 0.1, 'patch_length': 16, 'learning_rate': 0.0005, 'batch_size': 32}. Best is trial 48 with value: 0.15609563116137173.


Best trial: 48. Best value: 0.156096: 100%|██████████| 50/50 [2:01:48<00:00, 146.16s/it]



✅ Busca Optuna concluída!
⏱️ Tempo total: 2:01:48.100061
🏆 Melhor RMSE: 0.156096

🏆 MELHORES HIPERPARÂMETROS (Optuna):
features_idx             : 1
context_length           : 256
forecast_horizon         : 1
d_model                  : 128
num_attention_heads      : 8
num_hidden_layers        : 2
ffn_dim                  : 256
dropout                  : 0.1
patch_length             : 16
learning_rate            : 0.0005
batch_size               : 32
⚠️ Não foi possível gerar visualizações Optuna: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.
   Instale: pip install plotly kaleido


In [38]:
import pandas as pd

optuna_results = pd.read_csv('optuna_results.csv')
optuna_results.sort_values(by='eval_loss', ascending=False).head()

experiment_id                                                        48
features               ['Vol_lag_1', 'Vol_week_mean', 'Vol_month_mean']
num_features                                                          3
context_length                                                      256
forecast_horizon                                                      1
d_model                                                             128
num_attention_heads                                                   8
num_hidden_layers                                                     2
ffn_dim                                                             256
dropout                                                             0.1
patch_length                                                         16
learning_rate                                                    0.0005
batch_size                                                           32
train_loss                                                     0